In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import os
import socket

In [ ]:
import sys
from pathlib import Path
project_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "model_training").is_dir())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
ROOT_DATASET_DIR = project_root / "datasets"
print(f"Project Root: {project_root}")

In [4]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [5]:
import tiktoken 
import torch
import torch.nn as nn
from torch.nn import functional as F 

In [ ]:
from model_training.framework.model.load_model.gpt2.configs import load_model_config
from model_training.framework.model.load_model.gpt2.model import GPT
from model_training.framework.model.components.inference.inference import gen_sample
from model_training.framework.data.pre_train.causal_modelling.tinyshakespeare.data import return_data, prepare_one_batch
from model_training.framework.data.pre_train.causal_modelling.dataloader import DataLoaderLite
from model_training.framework.train.core.train import train_model
from model_training.framework.train.core.config import TrainConfig
from model_training.framework.train.utils.lr_scheduler import cosine_lr

In [7]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

### Loading Common Configs

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
num_return_sequences = 5
max_length = 30
config = load_model_config(max_length, "gpt2-small")

### Loading The Model

In [ ]:
# Set to True to download/load the pretrained GPT-2 checkpoint.
USE_PRETRAINED = False
model = GPT.from_pretrained(config, "gpt2") if USE_PRETRAINED else GPT(config)
model = model.to(device).eval()
print(f"Model loaded on {device}: {type(model).__name__}")

#### Doing a forward Pass

In [10]:
x = "Hello, I am a language model, not a program."
enc_x = tokenizer.encode(x)
print(enc_x, type(enc_x))

[15496, 11, 314, 716, 257, 3303, 2746, 11, 407, 257, 1430, 13] <class 'list'>


In [11]:
enc_x = torch.tensor(enc_x, dtype = torch.long)
enc_x = enc_x.unsqueeze(0).repeat(num_return_sequences, 1)
enc_x= enc_x.to(device)
print(enc_x.shape)

torch.Size([5, 12])


#### Implementing the Prefill and Decode

In [12]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
print(x)

Hello, I am a language model, not a program.


In [ ]:
output = gen_sample(model=model, input_batch=enc_x, max_new_tokens=10, context_len=max_length)

In [ ]:
output_list = output.tolist()
for i in range(num_return_sequences):
    decoded = tokenizer.decode(output_list[i])
    print(f"{i+1}>", decoded)
    print("=============================================")

### Loading and Preparing the Dataset

In [15]:
train_data = return_data(ROOT_DATASET_DIR, tokenizer)
len(train_data)

338025

In [16]:
SAMPLE_TOKENS = 25
sample_data = torch.tensor(train_data[:SAMPLE_TOKENS])
print(sample_data)

tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597,  2252,    11,
         3285,   502,  2740,    13,   198,   198,  3237,    25,   198,  5248,
          461,    11,  2740,    13,   198])


In [ ]:
x, y = prepare_one_batch(sample_data, batch=4, context_len=6)
print(x.shape, y.shape)

### Step - Training the Model

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class TrainComponents:
    optimizer: Any
    dataloader: DataLoaderLite
    scheduler: Any

In [ ]:
# TrainComponents uses the current DataLoaderLite and cosine scheduler.

In [ ]:
train_config = TrainConfig(num_steps=1, learning_rate=3e-4, device=device,
    total_batch_size=24, micro_batch_size=4, seq_length=6, accumulation_steps=1,
    max_lr=3e-4, min_lr=3e-5, warmup_steps=0, use_compile=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=train_config.learning_rate)
train_config

In [ ]:
dataloader = DataLoaderLite(sample_data.tolist(), batch_size=4, seq_len=6, tokenizer=tokenizer)
train_comp = TrainComponents(optimizer=optimizer, dataloader=dataloader, scheduler=cosine_lr)
train_comp

In [ ]:
# Opt in to a single training step when desired.
RUN_TRAINING = False
if RUN_TRAINING:
    model = train_model(model, train_components=train_comp, train_config=train_config)